# Architecture Scaling and Efficiency Study

Analyses DCF metrics across model spectrum using empirical data from notebooks 01-03.
No GPU required. Includes CS heatmap, efficiency frontier, scaling plots.

In [ ]:
OUTPUT_DIR = r'E:\decision_consistency_scaling'
import os, csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR,exist_ok=True)

In [ ]:
# Empirical results: (dataset, model, accuracy, LS, CD, CS, depth, params_M, inference_ms)
data=[('CIFAR-10','ResNet-18',0.876,0.694,0.129,0.782,18,11.7,1.2),('CIFAR-10','ResNet-50',0.865,0.719,0.142,0.789,50,25.6,2.1),('CIFAR-10','VGG-16',0.856,0.667,0.113,0.777,16,138.4,5.3),('CIFAR-10','MobileNetV2',0.874,0.685,0.133,0.776,53,3.5,0.8),('STL-10','ResNet-18',0.939,0.821,0.107,0.857,18,11.7,1.2),('STL-10','ResNet-50',0.969,0.851,0.105,0.873,50,25.6,2.1),('STL-10','VGG-16',0.936,0.800,0.084,0.838,16,138.4,5.3),('STL-10','MobileNetV2',0.935,0.823,0.117,0.854,53,3.5,0.8),('Intel-Scene','ResNet-18',0.574,0.813,0.081,0.866,18,11.7,1.2),('Intel-Scene','ResNet-50',0.526,0.799,0.096,0.848,50,25.6,2.1),('Intel-Scene','VGG-16',0.742,0.788,0.092,0.848,16,138.4,5.3),('Intel-Scene','MobileNetV2',0.599,0.773,0.095,0.839,53,3.5,0.8),('OCTMNIST','ResNet-18',0.470,0.768,0.046,0.861,18,11.7,1.2),('OCTMNIST','ResNet-50',0.526,0.746,0.058,0.847,50,25.6,2.1),('OCTMNIST','VGG-16',0.628,0.710,0.022,0.830,16,138.4,5.3),('OCTMNIST','MobileNetV2',0.513,0.717,0.080,0.815,53,3.5,0.8)]
models_unique=['ResNet-18','ResNet-50','VGG-16','MobileNetV2']; datasets_unique=['CIFAR-10','STL-10','Intel-Scene','OCTMNIST']
avg_cs={m:np.mean([cs for ds,model,acc,ls,cd,cs,*_ in data if model==m]) for m in models_unique}
print('Avg CS per model:'); [print(f'  {m:<14}: {v:.4f}') for m,v in avg_cs.items()]

In [ ]:
depths=[18,50,16,53]; params=[11.7,25.6,138.4,3.5]
cs_cifar=[next(cs for ds,model,acc,ls,cd,cs,*_ in data if ds=='CIFAR-10' and model==mn) for mn in models_unique]
cs_stl=[next(cs for ds,model,acc,ls,cd,cs,*_ in data if ds=='STL-10' and model==mn) for mn in models_unique]
fig,axes=plt.subplots(1,2,figsize=(12,5))
for cs_list,label in [(cs_cifar,'CIFAR-10'),(cs_stl,'STL-10')]:
    axes[0].scatter(depths,cs_list,s=120,label=label); axes[1].scatter(params,cs_list,s=120,label=label)
    for d,cs_v,mn in zip(depths,cs_list,models_unique): axes[0].annotate(mn[:6],(d,cs_v),fontsize=7,ha='center',va='bottom')
    for p,cs_v,mn in zip(params,cs_list,models_unique): axes[1].annotate(mn[:6],(p,cs_v),fontsize=7,ha='center',va='bottom')
axes[0].set_xlabel('Depth'); axes[0].set_ylabel('CS'); axes[0].set_title('CS vs Architecture Depth'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_xlabel('Params (M)'); axes[1].set_xscale('log'); axes[1].set_title('Efficiency: CS vs Model Size'); axes[1].legend(); axes[1].grid(alpha=0.3)
fig.suptitle('Architecture Scaling — DCF'); plt.tight_layout(); path=os.path.join(OUTPUT_DIR,'architecture_scaling.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)

In [ ]:
cs_matrix=np.array([[next(cs for ds,model,acc,ls,cd,cs,*_ in data if ds==dataset and model==mn) for dataset in datasets_unique] for mn in models_unique])
fig,ax=plt.subplots(figsize=(8,5))
im=ax.imshow(cs_matrix,cmap='RdYlGn',vmin=0.75,vmax=0.90)
ax.set_xticks(range(len(datasets_unique))); ax.set_yticks(range(len(models_unique)))
ax.set_xticklabels(datasets_unique,fontsize=10); ax.set_yticklabels(models_unique,fontsize=10)
for i in range(len(models_unique)):
    for j in range(len(datasets_unique)): ax.text(j,i,f'{cs_matrix[i,j]:.3f}',ha='center',va='center',fontsize=11)
plt.colorbar(im,ax=ax,label='Consistency Score'); ax.set_title('CS Heatmap — Models x Datasets',fontsize=12); plt.tight_layout()
path=os.path.join(OUTPUT_DIR,'cs_heatmap.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)

In [ ]:
print(f'{"Model":<14} {"CS_avg":>8} {"Params":>8} {"CS/sqrtP":>10} {"CS/ms":>10}'); print('-'*55)
rows=[]
for i,mn in enumerate(models_unique):
    cs_avg=avg_cs[mn]; p=params[i]; ms=[1.2,2.1,5.3,0.8][i]; eff_p=cs_avg/(p**0.5); eff_ms=cs_avg/ms
    print(f'{mn:<14} {cs_avg:>8.4f} {p:>8.1f} {eff_p:>10.4f} {eff_ms:>10.4f}')
    rows.append({'Model':mn,'CS_avg':round(cs_avg,4),'Params_M':p,'CS_per_sqrtP':round(eff_p,4),'CS_per_ms':round(eff_ms,4)})
csv_path=os.path.join(OUTPUT_DIR,'efficiency_scores.csv')
with open(csv_path,'w',newline='') as f: w=csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print('Saved:',csv_path)